## Installing Unsloth for Efficient Fine-Tuning



In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## Dataset Formatting using Llama-3 Chat Template

This step converts raw company-specific data into a format suitable for fine-tuning a Llama-3 model.

### Purpose
The goal is to structure the dataset in a conversational format so that the model learns:
- Company-specific acronyms
- Required structured JSON outputs
- Strict formatting rules

### Key Approach
- Uses `AutoTokenizer.apply_chat_template()` to ensure correct formatting
- Automatically inserts special tokens such as:
  - `<|begin_of_text|>`
  - `<|eot_id|>`
- Prevents formatting errors during training

In [ ]:
import json
from transformers import AutoTokenizer

def format_with_chat_template(raw_data_path, output_path, model_id="unsloth/llama-3-8b-Instruct"):
    """
    Converts raw JSON data into Llama-3 instruction format using the
    official tokenizer.apply_chat_template.

    This handles all special tokens (<|begin_of_text|>, <|eot_id|>, etc.)
    automatically and correctly.
    """
    print(f"Loading tokenizer for {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    with open(raw_data_path, "r") as f:
        raw_examples = json.load(f)

    formatted_dataset = []

    for item in raw_examples:
        # Extract fields
        acr = item['acronym']
        full = item['full']
        cat = item['category']
        prio = item['priority']
        act = item['action']

        # Define the conversation structure
        messages = [
            {
                "role": "system",
                "content": "You are a company syntax parser. Always return valid JSON only."
            },
            {
                "role": "user",
                "content": f"Explain {acr} using the company schema."
            },
            {
                "role": "assistant",
                "content": json.dumps({
                    "acronym": acr,
                    "expanded_form": full,
                    "category": cat,
                    "priority": prio,
                    "required_action": act,
                    "owner_role": f"{cat}_lead"
                })
            }
        ]

        # Apply the template
        # add_generation_prompt=False because we are providing the assistant response for training
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        formatted_dataset.append({"text": text})

    # Save as JSONL (required for SFTTrainer)
    with open(output_path, "w") as f:
        for entry in formatted_dataset:
            f.write(json.dumps(entry) + "\n")

    print(f"Success: Converted {len(formatted_dataset)} items to {output_path}")

if __name__ == "__main__":
    # Ensure raw_company_data.json exists before running
    format_with_chat_template("/content/trainData.jsonl", "company_syntax_50.jsonl")

Loading tokenizer for unsloth/llama-3-8b-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Success: Converted 50 items to company_syntax_50.jsonl


## LoRA Fine-Tuning with Unsloth

This cell performs supervised fine-tuning of a Llama-3 8B model using Unsloth and LoRA adapters.

### What This Step Does
- Loads the 4-bit quantized Llama-3 8B model
- Adds LoRA adapters for parameter-efficient fine-tuning
- Loads the formatted company syntax dataset
- Splits the dataset into training and evaluation sets
- Trains the model using `SFTTrainer`
- Saves only the trained LoRA adapter

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Configuration
model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
max_seq_length = 2048
dataset_file = "/content/company_syntax_50.jsonl"

# 2. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,

)

# 3. Add LoRA Adapters (Phase 3 Optimization)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# 4. Load Dataset and Split for Evaluation
# We split 10% of the data to use as a "test" set for the grader logic
full_dataset = load_dataset("json", data_files=dataset_file, split="train")
dataset_split = full_dataset.train_test_split(test_size=0.1)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# 5. Training Setup
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset, # Added evaluation dataset
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        eval_strategy = "steps", # Perform evaluation during training
        eval_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = "outputs",
        save_strategy = "no",
    ),
)

# 6. Run Training
print("Starting Fine-Tuning with Evaluation Split...")
trainer.train()

# 7. Save the Adapter
model.save_pretrained("company_syntax_lora_adapter")
tokenizer.save_pretrained("company_syntax_lora_adapter")
print("Training complete. Adapter saved to 'company_syntax_lora_adapter'")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Starting Fine-Tuning with Evaluation Split...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 45 | Num Epochs = 10 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
10,1.072403,1.016571
20,0.456117,0.584949
30,0.334527,0.543923
40,0.185425,0.559705
50,0.128588,0.575464
60,0.120305,0.588087


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training complete. Adapter saved to 'company_syntax_lora_adapter'


## Programmable JSON Grader (Reinforcement Signal)

This cell defines a programmable grader used in reinforcement fine-tuning (RFT).

In [ ]:
import json

def json_programmable_grader(model_output: str) -> float:
    """
    Phase 2: The Grader Loop.
    Checks if the model's assistant response is valid, parseable JSON.
    """
    try:
        # Step 1: Attempt to load the JSON
        # We handle potential leading/trailing whitespace or markdown blocks
        clean_text = model_output.strip().replace("```json", "").replace("```", "")
        json.loads(clean_text)

        # Reward 1.0 for valid JSON
        return 1.0
    except (json.JSONDecodeError, ValueError):
        # Penalty -1.0 for invalid JSON
        return -1.0


if __name__ == "__main__":
    # Test valid case
    sample_ok = '{"acronym": "QBR", "full": "Quarterly Business Review"}'
    print(f"Valid Sample Reward: {json_programmable_grader(sample_ok)}")

    # Test invalid case
    sample_bad = "QBR stands for Quarterly Business Review."
    print(f"Invalid Sample Reward: {json_programmable_grader(sample_bad)}")



Valid Sample Reward: 1.0
Invalid Sample Reward: -1.0


## GPU Memory Cleanup


In [ ]:
import torch
import gc

# Delete existing model and trainer to free VRAM
if "model" in globals():
    del model
if "trainer" in globals():
    del trainer
if "tokenizer" in globals():
    del tokenizer

# Force garbage collection and clear CUDA cache
gc.collect()
torch.cuda.empty_cache()

print("GPU Memory Cleared. You can now load the model for inference.")

GPU Memory Cleared. You can now load the model for inference.


## Inference Test with LoRA Adapter

This step loads the trained LoRA adapter and runs a test inference.

### What This Does
- Loads the fine-tuned model in 4-bit mode
- Formats a test prompt using the chat template
- Generates a response from the model
- Extracts and prints the assistant’s output

In [ ]:
from unsloth import FastLanguageModel
import json

# 1. Load the model and the adapter we just trained
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "company_syntax_lora_adapter",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # Enable 2x faster inference

# 2. Prepare a test prompt
test_acronym = "SOC2"

messages = [
    {"role": "system", "content": "You are a company syntax parser.Always return valid JSON only."},
    {"role": "user", "content": f"Explain {test_acronym} using the company schema."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 3. Generate
outputs = model.generate(input_ids = inputs, max_new_tokens = 128)
response = tokenizer.decode(outputs[0], skip_special_tokens = True)

# 4. Final Verification
assistant_response = response.split("assistant")[-1].strip()
print(f"--- Model Output for {test_acronym} ---")
print(assistant_response)

==((====))==  Unsloth 2026.5.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py

--- Model Output for SOC2 ---
{"acronym": "SOC2", "expanded_form": "Systems and Organization Controls 2", "category": "Security", "priority": "Critical", "required_action": "Prepare for annual compliance audit.", "owner_role": "Security_lead"}


## GGUF Export (Model Merging + Quantization)

This step loads the trained LoRA adapter, merges it with the base model, and exports it to GGUF format.

### What This Does
- Clears GPU memory to prevent OOM errors
- Loads the LoRA adapter in 4-bit mode (T4 optimized)
- Merges adapter weights into the base model
- Exports the model as a 4-bit GGUF file (`q4_k_m`)


In [ ]:
from unsloth import FastLanguageModel
import torch
import gc
import os

# 1. Aggressive Memory Cleanup
if "model" in globals(): del model
if "tokenizer" in globals(): del tokenizer
if "trainer" in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()

# 2. Load for merging with T4-optimized settings
# We use 'load_in_4bit=True' here to save VRAM, but Unsloth's GGUF
print("Loading model for GGUF export (T4 Optimized)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "company_syntax_lora_adapter",
    max_seq_length = 2048,
    dtype = None,               # Auto-detect (will use float16 on T4)
    load_in_4bit = True,
)

# 3. Merge and Save to GGUF
# quantization_method = "q4_k_m"
print("Merging and exporting to 4-bit GGUF...")


try:
    model.save_pretrained_gguf(
        "company_model_gguf",
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("Export Complete!")
except Exception as e:
    print(f" Error during GGUF export: {e}")


# 4. Final step: Show the generated file path
if os.path.exists("company_model_gguf"):
    files = os.listdir("company_model_gguf")
    print(f"\nFiles in export folder: {files}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model for GGUF export (T4 Optimized)...
==((====))==  Unsloth 2026.5.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Merging and exporting to 4-bit GGUF...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in company_model_gguf/tokenizer_config.json.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 10972.67it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [07:49<00:00, 117.42s/it]


Unsloth: Merge process complete. Saved to `/content/company_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['company_model_gguf_gguf/llama-3-8b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['company_model_gguf_gguf/llama-3-8b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model company_model_gguf_gguf/llama-3-8b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to company_model_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f company_model_gguf_gguf/Modelfile
Export Complete!

Files in export folder: ['config.json', '.cache', 'model-00001-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors.index.json', 'model-000

In [ ]:
!du -h --max-depth=1 /content | sort -h

4.0K	/content/outputs
148K	/content/.config
3.0M	/content/unsloth_compiled_cache
8.8M	/content/huggingface_tokenizers_cache
55M	/content/sample_data
177M	/content/company_syntax_lora_adapter
15G	/content/company_model_gguf
16G	/content


In [ ]:
from google.colab import files

# 1. Download the GGUF model file
# Double check the folder name; it's likely 'company_model_gguf_gguf'
model_path = "/content/company_model_gguf_gguf/llama-3-8b-instruct.Q4_K_M.gguf"
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# The !cp command is 'copy'
!cp "/content/company_model_gguf_gguf/llama-3-8b-instruct.Q4_K_M.gguf" "/content/drive/MyDrive/"